<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/ICHI2027_Phase39_SQL_Cohort_Replay_Drive_to_Drive_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICHI 2027 · Phase 39 — SQL source ledger, cohort replay and negative controls
**Google Drive → Google Drive | Phase 37 + 38 SHA chain | official Arm A stays BLOCKED**

This notebook implements an **executable relational staging reference**, *not an OMOP or MI-CDM target loader*. It keeps the original 19-person Phase 33 synthetic FHIR source separate from the 38 Phase 37 **post-hoc manufactured synthetic Procedure fixtures**. It verifies all six Phase 37 artifacts, all public Phase 38 artifacts and the original Phase 33 bundle against their frozen SHA-256 hashes before running SQL. If Phase 38 has not yet been run, run Phase 38 first; do not manufacture Phase 38 outputs.

It then creates five SQLite source-ledger tables plus a manufactured-fixture table, loads source data twice to check stable replay/idempotence, independently computes the *synthetic bilateral-hippocampus source-proxy eligibility* from raw FHIR and from SQL, checks all observation fields/source provenance, runs deliberately failing SQL integrity controls and exports public-safe outputs. The value-only illustration is **not** an official matched four-arm comparison; lacking laterality and timepoint, source-proxy membership from value-only rows is NOT EVALUABLE. No clinical MCI label is inferred. No private Athena data or selected concept ID is ever read by this notebook.

**Input requirement:** matching Phase 38 SHAREABLE + hashed public companions; exact matching Phase 37 SHAREABLE + six hashed companions; exact original Phase 33 FHIR bundle. Rename suffixes `(1)` are supported by SHA matching, but mixed runs are rejected. Set `DRIVE_RELATIVE_FOLDER` if your files live below MyDrive. Public Phase 39 outputs are saved **beside matching Phase 38 files**, never in `_PRIVATE_ATHENA`.

In [1]:
import os,json,csv,hashlib,sqlite3,datetime,copy,collections,math
from pathlib import Path
if os.environ.get('ICHI2027_DRIVE_ROOT'):
    DRIVE_ROOT=Path(os.environ['ICHI2027_DRIVE_ROOT'])
else:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT=Path('/content/drive/MyDrive')
DRIVE_RELATIVE_FOLDER=''  # For example: 'NeuroFHIR_ICHI2027'
ROOT=DRIVE_ROOT/DRIVE_RELATIVE_FOLDER if DRIVE_RELATIVE_FOLDER else DRIVE_ROOT
assert ROOT.is_dir(),f'Missing Drive directory: {ROOT}'
print('Input search root:',ROOT)

Mounted at /content/drive
Input search root: /content/drive/MyDrive


In [2]:
# Exact-SHA file discovery, with no private-folder traversal.
def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1<<20),b''):h.update(block)
    return h.hexdigest()
def discover(root,part):
    found=[]
    for folder,dirs,files in os.walk(root,onerror=lambda _:None):
        dirs[:]=[d for d in dirs if d not in {'_PRIVATE_ATHENA','.git','node_modules'} and not d.startswith('.')]
        for name in files:
            if name.startswith(part):found.append(Path(folder)/name)
    return sorted(found,key=lambda x:x.stat().st_mtime,reverse=True)
def companion(folder,fname,expected):
    stem=Path(fname).stem;suffix=Path(fname).suffix
    candidates=[p for p in folder.glob(stem+'*'+suffix) if p.is_file()]
    exact=[p for p in candidates if sha(p)==expected]
    if not exact:raise FileNotFoundError(f'Missing or modified {fname} (expected SHA {expected}).')
    return exact[0]

P38=None;errors=[]
for p in discover(ROOT,'ICHI2027_Phase38_SHAREABLE'):
    try:
        if p.suffix!='.json':continue
        q=json.loads(p.read_text('utf-8'))
        if q.get('phase')!='38_private_frozen_athena_and_human_binding_review_gate':continue
        assert q['tests']['total']==q['tests']['passed']==12
        companions={name:companion(p.parent,name,digest) for name,digest in q['public_artifact_sha256'].items()}
        # Phase 38 and Phase 37 must form the exact same source chain.
        pp=[x for x in p.parent.glob('ICHI2027_Phase37_SHAREABLE*.json') if x.is_file() and sha(x)==q['upstream']['phase37_shareable_sha256']]
        if not pp:raise FileNotFoundError('Matching Phase 37 SHAREABLE not in Phase 38 folder')
        P38=(p,q,companions,pp[0]);break
    except (KeyError,ValueError,AssertionError,OSError) as exc:errors.append((str(p),str(exc)))
if P38 is None:raise FileNotFoundError('Run Phase 38 first and keep its public outputs with Phase 37: '+str(errors[:4]))
P38_FILE,p38,p38files,P37_FILE=P38
p37=json.loads(P37_FILE.read_text('utf-8'))
assert p37['phase']=='37_posthoc_manufactured_synthetic_procedures_for_phase33_cohort'
assert p37['upstream']['imagewg_commit']==p38['upstream']['imagewg_commit']
assert p37['upstream']['omop55_commit']==p38['upstream']['omop55_commit']
p37files={name:companion(P37_FILE.parent,name,digest) for name,digest in p37['artifact_sha256'].items()}
assert len(p37files)==6 and len(p38files)==4
assert p38['phase37_verified']==dict(synthetic_people=19,synthetic_studies=38,
  manufactured_fixture_procedures=38,source_observations=148,
  observation_to_fixture_joins=148,negative_link_tests_passed=13)
assert p37['gaps']['original_phase29_vocab_gaps_blocked']==36
phase38_status=list(csv.DictReader(p38files['ICHI2027_Phase38_BINDING_REVIEW_STATUS.csv'].open(encoding='utf-8-sig',newline='')))
phase38_tests=list(csv.DictReader(p38files['ICHI2027_Phase38_GATE_TESTS.csv'].open(encoding='utf-8-sig',newline='')))
assert len(phase38_status)==19 and len(phase38_tests)==12
assert len({r['binding_key'] for r in phase38_status})==19
assert all(r['passed'].lower()=='true' for r in phase38_tests)

# Find original *unmodified* Phase 33 bundle by the exact Phase 37 parent SHA.
old_sha=p37['upstream']['phase33_original_fhir_sha256']
original_candidates=discover(ROOT,'ICHI2027_Phase33_NEW_SYNTHETIC_FHIR_BUNDLE')
original=next((p for p in original_candidates if p.suffix=='.json' and sha(p)==old_sha),None)
if original is None:
    raise FileNotFoundError('Copy your original Phase 33 NEW_SYNTHETIC_FHIR_BUNDLE.json to Google Drive; exact SHA required: '+old_sha)
original_bundle=json.loads(original.read_text('utf-8'))
patched_bundle=json.loads(p37files['ICHI2027_Phase37_SYNTHETIC_FHIR_WITH_PROCEDURE_FIXTURES.json'].read_text('utf-8'))
OUTDIR=P38_FILE.parent
print('Phase 38:',P38_FILE.name,'— 12/12 validated; all 4 public companions SHA-verified')
print('Phase 37:',P37_FILE.name,'— 6 companions SHA-verified')
print('Original Phase 33:',original.name,'— original SHA verified')

Phase 38: ICHI2027_Phase38_SHAREABLE.json — 12/12 validated; all 4 public companions SHA-verified
Phase 37: ICHI2027_Phase37_SHAREABLE.json — 6 companions SHA-verified
Original Phase 33: ICHI2027_Phase33_NEW_SYNTHETIC_FHIR_BUNDLE.json — original SHA verified


In [3]:
# Reconcile all ORIGINAL FHIR records independently of Phase 37 manufactured links.
def index_bundle(bundle):
    indexed={}
    for entry in bundle['entry']:
        r=entry['resource'];ref=r['resourceType']+'/'+r['id']
        assert ref not in indexed,'Duplicate FHIR resource '+ref
        indexed[ref]=r
    return indexed
src=index_bundle(original_bundle);patched=index_bundle(patched_bundle)
source_counts=collections.Counter(r['resourceType'] for r in src.values())
patched_counts=collections.Counter(r['resourceType'] for r in patched.values())
assert source_counts=={'Observation':148,'Provenance':148,'ImagingStudy':38,'Device':19,'Patient':19}
assert patched_counts=={'Observation':148,'Provenance':186,'ImagingStudy':38,'Device':19,'Patient':19,'Procedure':38}
for ref,r in src.items():
    rr=copy.deepcopy(patched[ref])
    if r['resourceType']=='ImagingStudy':
        assert 'procedureReference' in rr
        del rr['procedureReference']
    assert rr==r,'Original raw FHIR source changed in Phase37: '+ref
assert not any(r.get('gender') or r.get('birthDate') for r in src.values() if r['resourceType']=='Patient')
assert p37['source']['original_source_procedure_count']==0
# Verify each manufactured Procedure and provenance pair against its original study.
links=list(csv.DictReader(p37files['ICHI2027_Phase37_SYNTHETIC_PROCEDURE_LINKS.csv'].open(encoding='utf-8-sig',newline='')))
joins=list(csv.DictReader(p37files['ICHI2027_Phase37_SOURCE_TO_FIXTURE_JOIN.csv'].open(encoding='utf-8-sig',newline='')))
assert len(links)==38 and len(joins)==148
fixture_by_study={r['study_ref']:r for r in links}
assert len(fixture_by_study)==38
for study,link in fixture_by_study.items():
    assert patched[study]['procedureReference']['reference']==link['procedure_ref']
    assert link['fixture_kind']=='POST_HOC_MANUFACTURED_SYNTHETIC_ONLY'
    assert link['clinical_source_procedure_attested'].lower()=='false'
    proc=patched[link['procedure_ref']]
    assert proc['subject']['reference']==link['person_ref']
    assert proc['performedDateTime']==src[study]['started']==link['study_started']
    assert 'manufactured-synthetic-source-linked-test' in [x['code'] for x in proc['meta']['tag']]
    prov=patched[link['provenance_ref']]
    assert prov['target']==[{'reference':link['procedure_ref']}]
    assert {'role':'derivation','what':{'reference':study}} in prov['entity']
print('Original resource-level equality preserved:',len(src),'FHIR resources; 38 explicit synthetic fixtures checked')

Original resource-level equality preserved: 372 FHIR resources; 38 explicit synthetic fixtures checked


In [4]:
# Original FHIR is the sole source of patient, imaging, biomarker and provenance facts.
# Fixture identifiers come ONLY from the independently tagged Phase37 companion.
def ext_value(obs,ending,key):
    matches=[e[key] for e in obs.get('extension',[]) if e.get('url','').endswith(ending) and key in e]
    assert len(matches)==1,(obs['id'],ending,matches)
    return matches[0]
def one_ref(obj,key):
    arr=obj[key];assert len(arr)==1
    return arr[0]['reference']
person_rows=[{'person_ref':r} for r in sorted(src) if r.startswith('Patient/')]
device_rows=[]
for ref,r in sorted(src.items()):
    if r['resourceType']!='Device':continue
    ver=r.get('version',[]);assert len(ver)==1 and 'value' in ver[0]
    device_rows.append({'device_ref':ref,'algorithm_name':r['deviceName'][0]['name'],'algorithm_version':ver[0]['value']})
study_rows=[]
for ref,r in sorted(src.items()):
    if r['resourceType']!='ImagingStudy':continue
    ident=[x['value'] for x in r['identifier'] if x.get('system')=='urn:dicom:uid']
    assert len(ident)==1 and len(r['series'])==1
    s=r['series'][0]
    study_rows.append({'study_ref':ref,'person_ref':r['subject']['reference'],
                       'study_started':r['started'],'synthetic_study_uid':ident[0],
                       'synthetic_series_uid':s['uid'],'modality_source_code':s['modality']['code'],
                       'uid_source_attested':0})
fixture_rows=[]
for study,l in sorted(fixture_by_study.items()):
    fixture_rows.append({'procedure_ref':l['procedure_ref'],'study_ref':study,
       'person_ref':l['person_ref'],'provenance_ref':l['provenance_ref'],
       'fixture_kind':l['fixture_kind'],'real_procedure_attested':0})
obs_rows=[];prov_rows=[]
for ref,r in sorted(src.items()):
    if r['resourceType']!='Observation':continue
    person=r['subject']['reference'];study=one_ref(r,'focus');device=r['device']['reference']
    body=r['bodySite']['coding'];codes=r['code']['coding'];assert len(body)==len(codes)==1
    assert body[0]['code'].startswith('SYN-')
    region_code=codes[0]['code'];laterality=body[0]['code'].split('-')[-1]
    assert region_code in {'hippocampus-volume','entorhinal-volume'} and laterality in {'LEFT','RIGHT'}
    qty=r['valueQuantity']; assert qty['unit']==qty['code']=='mm3' and isinstance(qty['value'],(int,float))
    timepoint=ext_value(r,'timepoint-label','valueString')
    assert timepoint in {'baseline','12-month'}
    trace=ext_value(r,'trace-state','valueCode');rule=ext_value(r,'trace-rule-version','valueString')
    provref='Provenance/prov-'+r['id'];assert provref in src
    provenance=src[provref];assert provenance['target']==[{'reference':ref}]
    ents=provenance['entity']
    assert {'role':'source','what':{'reference':study}} in ents
    assert {'role':'source','what':{'reference':device}} in ents
    assert src[study]['subject']['reference']==person
    assert len([x for x in src.values() if x['resourceType']=='ImagingStudy' and x['subject']['reference']==person])>=1
    obs_rows.append({'obs_ref':ref,'person_ref':person,'study_ref':study,'device_ref':device,
       'biomarker_code':region_code,'anatomy_source_code':body[0]['code'],
       'laterality':laterality,'timepoint':timepoint,'effective_utc':r['effectiveDateTime'],
       'value_mm3':float(qty['value']),'unit_code':qty['code'],'trace_state':trace,
       'trace_version':rule,'provenance_ref':provref})
    prov_rows.append({'provenance_ref':provref,'obs_ref':ref,'study_ref':study,'device_ref':device})
assert (len(person_rows),len(device_rows),len(study_rows),len(fixture_rows),len(obs_rows),len(prov_rows))==(19,19,38,38,148,148)
assert len(set(r['synthetic_study_uid'] for r in study_rows))==38
assert len(set(r['synthetic_series_uid'] for r in study_rows))==38
assert {r['obs_ref'] for r in obs_rows}=={x['observation_ref'] for x in joins}
for join in joins:
    row=next(x for x in obs_rows if x['obs_ref']==join['observation_ref'])
    l=fixture_by_study[row['study_ref']]
    assert join['person_ref']==row['person_ref']==l['person_ref']
    assert join['synthetic_procedure_ref']==l['procedure_ref']
    assert join['synthetic_procedure_provenance_ref']==l['provenance_ref']
print('Built ORIGINAL-SOURCE ledger manifest: 19 persons, 38 studies, 148 observations; 38 separately tagged fixtures')

Built ORIGINAL-SOURCE ledger manifest: 19 persons, 38 studies, 148 observations; 38 separately tagged fixtures


In [5]:
# Strict SQLite relational schema: intentionally NOT named as official OMOP/MI-CDM target tables.
SCHEMA="\nPRAGMA foreign_keys=ON;\nCREATE TABLE source_person (\n    person_ref TEXT PRIMARY KEY\n);\nCREATE TABLE source_device (\n    device_ref TEXT PRIMARY KEY,\n    algorithm_name TEXT NOT NULL,\n    algorithm_version TEXT NOT NULL\n);\nCREATE TABLE source_study (\n    study_ref TEXT PRIMARY KEY,\n    person_ref TEXT NOT NULL,\n    study_started TEXT NOT NULL,\n    synthetic_study_uid TEXT NOT NULL UNIQUE,\n    synthetic_series_uid TEXT NOT NULL UNIQUE,\n    modality_source_code TEXT NOT NULL,\n    uid_source_attested INTEGER NOT NULL CHECK(uid_source_attested=0),\n    UNIQUE(study_ref,person_ref),\n    FOREIGN KEY(person_ref) REFERENCES source_person(person_ref)\n);\nCREATE TABLE synthetic_procedure_fixture (\n    procedure_ref TEXT PRIMARY KEY,\n    study_ref TEXT NOT NULL UNIQUE,\n    person_ref TEXT NOT NULL,\n    provenance_ref TEXT NOT NULL UNIQUE,\n    fixture_kind TEXT NOT NULL CHECK(fixture_kind='POST_HOC_MANUFACTURED_SYNTHETIC_ONLY'),\n    real_procedure_attested INTEGER NOT NULL CHECK(real_procedure_attested=0),\n    FOREIGN KEY(study_ref,person_ref) REFERENCES source_study(study_ref,person_ref)\n);\nCREATE TABLE source_observation (\n    obs_ref TEXT PRIMARY KEY,\n    person_ref TEXT NOT NULL,\n    study_ref TEXT NOT NULL,\n    device_ref TEXT NOT NULL,\n    biomarker_code TEXT NOT NULL CHECK(biomarker_code IN ('hippocampus-volume','entorhinal-volume')),\n    anatomy_source_code TEXT NOT NULL,\n    laterality TEXT NOT NULL CHECK(laterality IN ('LEFT','RIGHT')),\n    timepoint TEXT NOT NULL CHECK(timepoint IN ('baseline','12-month')),\n    effective_utc TEXT NOT NULL,\n    value_mm3 REAL NOT NULL CHECK(value_mm3>0),\n    unit_code TEXT NOT NULL CHECK(unit_code='mm3'),\n    trace_state TEXT NOT NULL,\n    trace_version TEXT NOT NULL,\n    provenance_ref TEXT NOT NULL UNIQUE,\n    FOREIGN KEY(person_ref) REFERENCES source_person(person_ref),\n    FOREIGN KEY(study_ref,person_ref) REFERENCES source_study(study_ref,person_ref),\n    FOREIGN KEY(device_ref) REFERENCES source_device(device_ref),\n    CHECK ((anatomy_source_code='SYN-HIP-LEFT' AND biomarker_code='hippocampus-volume' AND laterality='LEFT') OR\n           (anatomy_source_code='SYN-HIP-RIGHT' AND biomarker_code='hippocampus-volume' AND laterality='RIGHT') OR\n           (anatomy_source_code='SYN-ENT-LEFT' AND biomarker_code='entorhinal-volume' AND laterality='LEFT') OR\n           (anatomy_source_code='SYN-ENT-RIGHT' AND biomarker_code='entorhinal-volume' AND laterality='RIGHT'))\n);\nCREATE TABLE source_observation_provenance (\n    provenance_ref TEXT PRIMARY KEY,\n    obs_ref TEXT NOT NULL UNIQUE,\n    study_ref TEXT NOT NULL,\n    device_ref TEXT NOT NULL,\n    FOREIGN KEY(obs_ref) REFERENCES source_observation(obs_ref),\n    FOREIGN KEY(study_ref) REFERENCES source_study(study_ref),\n    FOREIGN KEY(device_ref) REFERENCES source_device(device_ref)\n);\nCREATE TRIGGER observation_provenance_matches BEFORE INSERT ON source_observation_provenance\nBEGIN\n    SELECT CASE WHEN NOT EXISTS (\n        SELECT 1 FROM source_observation o WHERE o.obs_ref=NEW.obs_ref\n        AND o.provenance_ref=NEW.provenance_ref AND o.study_ref=NEW.study_ref\n        AND o.device_ref=NEW.device_ref\n    ) THEN RAISE(ABORT,'provenance mismatch with FHIR source observation') END;\nEND;\n"
con=sqlite3.connect(':memory:')
con.execute('PRAGMA foreign_keys=ON')
con.executescript(SCHEMA)
DATASETS=[('source_person',person_rows),('source_device',device_rows),('source_study',study_rows),
          ('synthetic_procedure_fixture',fixture_rows),('source_observation',obs_rows),
          ('source_observation_provenance',prov_rows)]
def load_once(conn,sets=DATASETS):
    # Idempotence: repeat data is accepted ONLY when payload is byte-for-byte equal to existing rows.
    with conn:
        for table,rows in sets:
            assert table in {x[0] for x in DATASETS}
            if not rows:continue
            cols=list(rows[0]); cols_sql=','.join(cols);qmarks=','.join('?'*len(cols))
            pk={'source_person':'person_ref','source_device':'device_ref','source_study':'study_ref',
                'synthetic_procedure_fixture':'procedure_ref','source_observation':'obs_ref',
                'source_observation_provenance':'provenance_ref'}[table]
            for row in rows:
                old=conn.execute('SELECT '+cols_sql+' FROM '+table+' WHERE '+pk+'=?',(row[pk],)).fetchone()
                val=tuple(row[k] for k in cols)
                if old is not None:
                    if old!=val:raise ValueError('Conflicting replay payload: '+table+' '+row[pk])
                else:conn.execute('INSERT INTO '+table+'('+cols_sql+') VALUES('+qmarks+')',val)

def counts(conn):return {t:conn.execute('SELECT COUNT(*) FROM '+t).fetchone()[0] for t,_ in DATASETS}
load_once(con);first_counts=counts(con);load_once(con);assert counts(con)==first_counts
assert first_counts=={'source_person':19,'source_device':19,'source_study':38,
 'synthetic_procedure_fixture':38,'source_observation':148,'source_observation_provenance':148}
assert con.execute('PRAGMA foreign_key_check').fetchall()==[]
print('Two SQL loads: stable row counts',first_counts)

Two SQL loads: stable row counts {'source_person': 19, 'source_device': 19, 'source_study': 38, 'synthetic_procedure_fixture': 38, 'source_observation': 148, 'source_observation_provenance': 148}


In [6]:
# Independent ORIGINAL FHIR vs SQL canonical row comparison — no downstream fixture is truth.
COLUMNS=list(obs_rows[0]);sql_rows={r[0]:dict(zip(COLUMNS,r)) for r in con.execute('SELECT '+','.join(COLUMNS)+' FROM source_observation')}
recon=[]
for raw in obs_rows:
    saved=sql_rows.get(raw['obs_ref']);mismatches=[k for k in COLUMNS if saved is None or raw[k]!=saved[k]]
    recon.append({'observation_ref':raw['obs_ref'],'source_fields_compared':len(COLUMNS),
      'exact_source_sql_match':not mismatches,'mismatched_fields':';'.join(mismatches)})
assert len(recon)==148 and all(r['exact_source_sql_match'] for r in recon)
SQL_COHORT="\nSELECT p.person_ref,\n  COUNT(DISTINCT CASE WHEN o.biomarker_code='hippocampus-volume'\n     AND o.provenance_ref=v.provenance_ref\n     THEN o.timepoint||':'||o.laterality END) AS bilateral_landmarks\nFROM source_person p\nLEFT JOIN source_observation o ON o.person_ref=p.person_ref\nLEFT JOIN source_observation_provenance v ON v.obs_ref=o.obs_ref\nGROUP BY p.person_ref ORDER BY p.person_ref;\n"
cohort_sql={ref:n for ref,n in con.execute(SQL_COHORT)}
cohort_source={}
for person in sorted(r['person_ref'] for r in person_rows):
    landmarks={x['timepoint']+':'+x['laterality'] for x in obs_rows
               if x['person_ref']==person and x['biomarker_code']=='hippocampus-volume'
               and x['provenance_ref'] in {p['provenance_ref'] for p in prov_rows}}
    cohort_source[person]=len(landmarks)
cohort_rows=[{'person_ref':p,'source_landmark_count':cohort_source[p],
   'sql_landmark_count':cohort_sql[p],
   'source_proxy_eligible':cohort_source[p]==4,
   'sql_proxy_eligible':cohort_sql[p]==4,
   'exact_agreement':cohort_source[p]==cohort_sql[p],
   'diagnosis_attested':False} for p in sorted(cohort_source)]
assert len(cohort_rows)==19 and all(r['exact_agreement'] for r in cohort_rows)
assert sum(r['source_proxy_eligible'] for r in cohort_rows)==15
assert sum(not r['source_proxy_eligible'] for r in cohort_rows)==4
# Opaque-key + scalar value alone has no supported patient, side or visit reconstruction.
value_only=[{'opaque_observation_key':i+1,'scalar_value':r['value_mm3']} for i,r in enumerate(obs_rows)]
assert len(value_only)==148 and all(set(r)=={'opaque_observation_key','scalar_value'} for r in value_only)
print('ORIGINAL FHIR vs SQL: 148/148 selected source rows exact; 19/19 source-proxy cohort agreements (15 positive, 4 negative)')
print('Value-only cohort status: NOT_EVALUABLE (person/laterality/timepoint absent); this is NOT an official B arm')

ORIGINAL FHIR vs SQL: 148/148 selected source rows exact; 19/19 source-proxy cohort agreements (15 positive, 4 negative)
Value-only cohort status: NOT_EVALUABLE (person/laterality/timepoint absent); this is NOT an official B arm


In [7]:
# Corruption controls: use isolated clones; do not alter the verified production in-memory ledger.
def clone():
    c=sqlite3.connect(':memory:');c.execute('PRAGMA foreign_keys=ON');con.backup(c)
    return c
def reject(name,action):
    c=clone();caught=False;reason=''
    try:action(c)
    except (sqlite3.IntegrityError,sqlite3.OperationalError,ValueError,AssertionError) as e:
        caught=True;reason=type(e).__name__+': '+str(e)[:130]
    finally:c.close()
    return {'test':name,'corruption_rejected':caught,'reason':reason}
def corrupt_replay(c):
    bad=copy.deepcopy(obs_rows[0]);bad['value_mm3']+=11.0
    load_once(c,[('source_observation',[bad])])
def wrong_person(c):
    row=copy.deepcopy(obs_rows[0]);row['obs_ref']='Observation/new-fabricated-p39';row['provenance_ref']='Provenance/new-fabricated-p39'
    row['person_ref']=next(p['person_ref'] for p in person_rows if p['person_ref']!=row['person_ref'])
    load_once(c,[('source_observation',[row])])
def wrong_source_prov(c):
    row=copy.deepcopy(prov_rows[0]);row['provenance_ref']='Provenance/new-wrong-p39';row['obs_ref']=obs_rows[1]['obs_ref']
    load_once(c,[('source_observation_provenance',[row])])
def duplicate_uid(c):
    row=copy.deepcopy(study_rows[0]);row['study_ref']='ImagingStudy/new-collision-p39'
    load_once(c,[('source_study',[row])])
def missing_study(c):
    row=copy.deepcopy(obs_rows[0]);row['obs_ref']='Observation/new-missingstudy-p39';row['provenance_ref']='Provenance/new-missingstudy-p39'
    row['study_ref']='ImagingStudy/no-such-p39'
    load_once(c,[('source_observation',[row])])
def wrong_laterality(c):
    row=copy.deepcopy(obs_rows[0]);row['obs_ref']='Observation/new-wrongside-p39';row['provenance_ref']='Provenance/new-wrongside-p39'
    row['laterality']='RIGHT' if row['laterality']=='LEFT' else 'LEFT'
    load_once(c,[('source_observation',[row])])
def bad_unit(c):
    row=copy.deepcopy(obs_rows[0]);row['obs_ref']='Observation/new-badunit-p39';row['provenance_ref']='Provenance/new-badunit-p39'
    row['unit_code']='cm3'
    load_once(c,[('source_observation',[row])])
def false_clinical_procedure(c):
    # Isolate the *specific* CHECK constraint, rather than coincidentally failing a UNIQUE key.
    c.execute('UPDATE synthetic_procedure_fixture SET real_procedure_attested=1 WHERE procedure_ref=?',
              (fixture_rows[0]['procedure_ref'],))
def missing_person(c):
    row=copy.deepcopy(study_rows[0]);row['study_ref']='ImagingStudy/new-missingperson-p39';row['synthetic_study_uid']='urn:oid:2.25.1';row['synthetic_series_uid']='2.25.2';row['person_ref']='Patient/missing-p39'
    load_once(c,[('source_study',[row])])
def empty_timepoint(c):
    row=copy.deepcopy(obs_rows[0]);row['obs_ref']='Observation/new-badvisit-p39';row['provenance_ref']='Provenance/new-badvisit-p39';row['timepoint']='unknown'
    load_once(c,[('source_observation',[row])])
def invalid_amount(c):
    row=copy.deepcopy(obs_rows[0]);row['obs_ref']='Observation/new-badvalue-p39';row['provenance_ref']='Provenance/new-badvalue-p39';row['value_mm3']=-3
    load_once(c,[('source_observation',[row])])
def detect_raw_tamper(c):
    # Verify true raw-byte SHA first, then change one confirmed observation-value byte.
    raw=original.read_bytes();assert hashlib.sha256(raw).hexdigest()==old_sha
    marker=b'"value": 3553';assert marker in raw,'Fixture-specific negative-control marker missing'
    changed=raw.replace(marker,b'"value": 3554',1)
    assert len(changed)==len(raw) and changed!=raw
    if hashlib.sha256(changed).hexdigest()!=old_sha:
        raise ValueError('Altered original raw FHIR bytes rejected by exact SHA')
negative=[reject(n,f) for n,f in [
    ('conflicting_observation_replay',corrupt_replay),('cross_person_study_foreign_key',wrong_person),
    ('mismatched_provenance_trigger',wrong_source_prov),('duplicated_study_uid',duplicate_uid),
    ('missing_imaging_study_fk',missing_study),('contradictory_laterality',wrong_laterality),
    ('wrong_volume_unit',bad_unit),('manufactured_fixture_misrepresented_as_clinical',false_clinical_procedure),
    ('missing_person_fk',missing_person),('illegal_timepoint',empty_timepoint),
    ('invalid_negative_volume',invalid_amount),('modified_original_fhir_hash',detect_raw_tamper)]]
assert len(negative)==12 and all(t['corruption_rejected'] for t in negative),[t for t in negative if not t['corruption_rejected']]
assert counts(con)==first_counts and con.execute('PRAGMA foreign_key_check').fetchall()==[]
print('Prespecified negative SQL/source controls:',sum(x['corruption_rejected'] for x in negative),'/',len(negative))

Prespecified negative SQL/source controls: 12 / 12


In [8]:
# Official target readiness is a documented list of BLOCKERS, not invented concept IDs.
field_evidence=list(csv.DictReader(p37files['ICHI2027_Phase37_ARM_A_FIELD_EVIDENCE.csv'].open(encoding='utf-8-sig',newline='')))
assert len(field_evidence)==81
status_counts=collections.Counter(r['status'] for r in phase38_status)
approved_public=sum(r['status']=='REVIEWED_SNAPSHOT_CHECKED_SYNTHETIC_ONLY' for r in phase38_status)
inapplicable_public=sum(r['status']=='REVIEWED_INAPPLICABLE_SOURCE_ABSENT' for r in phase38_status)
assert approved_public==p38['private_gate']['manual_snapshot_checked_approvals']
assert inapplicable_public==p38['private_gate']['reviewed_inapplicable_gender_keys']
blocking=[
 ('authorized_frozen_athena_with_valid_manifest',p38['private_gate']['snapshot_manifest_valid'],
    'Run Phase38 with authorized, private SHA-matched CONCEPT and its manifest'),
 ('19_binding_human_adjudication',approved_public+inapplicable_public==19,
    'Independently review each supported binding; source-absent gender keys must be documented as not applicable'),
 ('original_36_vocabulary_gap_instances_resolved',p38['private_gate']['original_36_vocabulary_gaps_declared_resolved'],
    'Evidence-review and independently resolve all distinct original gap instances'),
 ('source_supported_clinical_procedures',False,
    'Only manufactured synthetic fixtures exist; do not imply real clinical Procedure evidence'),
 ('real_dicom_uid_source_attested',False,
    'All current DICOM-style UIDs are synthetic identifiers'),
 ('missing_gender_and_birthdate_policy_independently_approved',False,
    'No source gender or birth dates; document policy without inventing facts'),
 ('official_MICDM_load_executed',False,'No official target load performed'),
 ('original_unmodified_generic_TFL_armC_executed',False,'Frozen original generic v15 is not portable to MRI as-is'),
 ('externally_independent_challenge_authored',False,'Developer-generated synthetic tests are not external validation'),
 ('real_authorized_MRI_processed',False,'No real MRI or ADNI study processed')]
readiness=[{'gate':g,'passed':bool(ok),'action':action} for g,ok,action in blocking]
assert all(not x['passed'] for x in readiness if x['gate'] in {'source_supported_clinical_procedures','official_MICDM_load_executed'})
print('Phase38 snapshot verified:',p38['private_gate']['snapshot_manifest_valid'],'approved (private IDs never read):',approved_public)
print('Official Arm A:', 'NOT_EVALUABLE — mandatory source, vocabulary and official load gates are incomplete')

Phase38 snapshot verified: False approved (private IDs never read): 0
Official Arm A: NOT_EVALUABLE — mandatory source, vocabulary and official load gates are incomplete


In [9]:
# Public-safe Drive artifacts: no private Athena paths, reviewer names or selected concept IDs.
PREFIX='ICHI2027_Phase39_'
def save_csv(filename,rows,fields):
    p=OUTDIR/(PREFIX+filename+'.csv')
    with p.open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=fields);w.writeheader();w.writerows(rows)
    return p
CNT=save_csv('SQL_STAGING_COUNTS',[{'table':k,'after_first_load':v,'after_second_load':counts(con)[k]} for k,v in first_counts.items()],
             ['table','after_first_load','after_second_load'])
REC=save_csv('SOURCE_SQL_RECONCILIATION',recon,['observation_ref','source_fields_compared','exact_source_sql_match','mismatched_fields'])
COH=save_csv('SOURCE_COHORT_AGREEMENT',cohort_rows,list(cohort_rows[0]))
NEG=save_csv('NEGATIVE_SQL_TESTS',negative,list(negative[0]))
GAT=save_csv('ARM_A_READINESS',readiness,list(readiness[0]))
SCRIPTPATH=OUTDIR/(PREFIX+'SOURCE_LEDGER_SCHEMA.sql')
SCRIPTPATH.write_text('-- SOURCE-LEDGER REFERENCE ONLY. NOT OMOP, MI-CDM, OR CLINICAL MAPPING.\n'+SCHEMA,encoding='utf-8')
DBPATH=OUTDIR/(PREFIX+'SYNTHETIC_SOURCE_LEDGER.sqlite')
if DBPATH.exists():DBPATH.unlink()  # The entire DB can be deterministically rebuilt from exact SHA-backed input.
con.execute('VACUUM')
public_db=sqlite3.connect(DBPATH)
con.backup(public_db);public_db.commit();public_db.close()
check=sqlite3.connect(DBPATH)
check.execute('PRAGMA foreign_keys=ON')
assert counts(check)==first_counts and check.execute('PRAGMA foreign_key_check').fetchall()==[]
check.close()
outputs=[CNT,REC,COH,NEG,GAT,SCRIPTPATH,DBPATH]
report={'project':'ICHI2027_neuroimaging_fhir_omop_fidelity',
 'phase':'39_sql_source_ledger_cohort_replay_and_integrity',
 'run_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
 'privacy':'Public synthetic only; no private Athena records, selected concept IDs, reviewers, clinical patient identifiers or private paths',
 'input_sha256':{'phase38_shareable':sha(P38_FILE),'phase37_shareable':sha(P37_FILE),
    'original_phase33_fhir_bundle':sha(original),'phase37_patched_bundle':sha(p37files['ICHI2027_Phase37_SYNTHETIC_FHIR_WITH_PROCEDURE_FIXTURES.json'])},
 'method':{'sql_engine':'Python SQLite','schema':'NON_STANDARD_SOURCE_LEDGER_NOT_OMOP_OR_MICDM',
    'upstream_raw_fhir_resource_level_equality':True,'relational_replay_count':2,
    'athena_private_files_read':False,'official_armA_loaded':False,
    'value_only_cohort_query':'NOT_EVALUABLE_MISSING_PERSON_LATERALITY_TIMEPOINT',
    'protocol':'Phase36 versioned internal future contract unchanged'},
 'synthetic_fixture':{'people':19,'original_imaging_studies':38,'original_observations':148,
    'separate_manufactured_procedures':38,'source_clinical_procedures_attested':0,
    'real_dicom_uids_attested':0,'source_mci_attested':False},
 'results':{'sql_table_counts':first_counts,'exact_source_sql_observations':148,
    'source_sql_cohort_agreement_people':19,'source_proxy_positive':15,'source_proxy_negative':4,
    'sql_negative_controls_detected':sum(x['corruption_rejected'] for x in negative),
    'sql_negative_controls_total':len(negative),'sql_idempotent_double_replay':True,
    'phase38_public_gate_tests_verified':12},
 'vocabulary_status':{'snapshot_manifest_valid':p38['private_gate']['snapshot_manifest_valid'],
    'bindings_snapshot_checked_approved':approved_public,
    'bindings_reviewed_source_inapplicable':inapplicable_public,
    'binding_keys_total':19,'original_vocabulary_gap_instances_blocked':36},
 'gates':{'official_micdm_load':False,'official_omop_etl':False,'full_A_B_C_D':False,
    'real_mri_rows':0,'independently_blinded_external_validation':False},
 'artifact_sha256':{p.name:sha(p) for p in outputs},
 'next_gate':'Complete authorized private frozen Athena + independent 19-binding review; then execute an official target-syntactic Arm A loader, official schema FK/cardinality/SQL replay and matched cohort queries. Never confuse this source-ledger reference with official target validation.'}
SHARE=OUTDIR/(PREFIX+'SHAREABLE.json');SHARE.write_text(json.dumps(report,indent=2)+'\n',encoding='utf-8')
assert SHARE.is_file() and len(report['artifact_sha256'])==7
print('PHASE 39 LOCAL/DRIVE RUN COMPLETE: source-ledger counts stable on two loads; 148/148 original FHIR matches; 19/19 proxy cohort agreement; 12/12 corruption controls')
print('Outputs:',OUTDIR)
for p in [SHARE,*outputs]:print(' ',p.name)
print('Athena and official target still BLOCKED unless separately independently reviewed and implemented.')

PHASE 39 LOCAL/DRIVE RUN COMPLETE: source-ledger counts stable on two loads; 148/148 original FHIR matches; 19/19 proxy cohort agreement; 12/12 corruption controls
Outputs: /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs
  ICHI2027_Phase39_SHAREABLE.json
  ICHI2027_Phase39_SQL_STAGING_COUNTS.csv
  ICHI2027_Phase39_SOURCE_SQL_RECONCILIATION.csv
  ICHI2027_Phase39_SOURCE_COHORT_AGREEMENT.csv
  ICHI2027_Phase39_NEGATIVE_SQL_TESTS.csv
  ICHI2027_Phase39_ARM_A_READINESS.csv
  ICHI2027_Phase39_SOURCE_LEDGER_SCHEMA.sql
  ICHI2027_Phase39_SYNTHETIC_SOURCE_LEDGER.sqlite
Athena and official target still BLOCKED unless separately independently reviewed and implemented.


## Upload after running

Upload these public-safe files to the conversation: `ICHI2027_Phase39_SHAREABLE.json`, `ICHI2027_Phase39_SQL_STAGING_COUNTS.csv`, `ICHI2027_Phase39_SOURCE_COHORT_AGREEMENT.csv` and `ICHI2027_Phase39_NEGATIVE_SQL_TESTS.csv`. You may also upload `SOURCE_SQL_RECONCILIATION.csv` for row-level review. Keep `_PRIVATE_ATHENA` private. The Phase 39 `.sqlite` is a **source-ledger reference**, not a validated official OMOP/MI-CDM implementation.